# Bài 2 (Trang 15-17) - Dữ liệu Titanic
## Câu 8: Tạo thuộc tính mới age_class và fare_per_person

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

df = pd.read_csv("../Dataset/titanic.csv")

### Các bước xử lý của câu 1 → 7 (chuẩn bị dữ liệu)

In [3]:
df["relatives"] = df["sibsp"] + df["parch"]
df["not_alone"] = (df["relatives"] > 0).astype(int)  # 1: đi cùng người thân, 0: đi một mình

# Tách ký tự đầu của cabin (boong tàu) trước khi xóa cabin, thiếu -> 'U' (Unknown)
deck_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "T": 8, "U": 0}
df["deck"] = df["cabin"].fillna("U").str[0].map(deck_map).fillna(0).astype(int)

# Xóa các thuộc tính không cần thiết
df = df.drop(columns=["body", "cabin", "boat", "home.dest", "ticket"])

# Điền giá trị thiếu cho age: sinh ngẫu nhiên trong [mean - std, mean + std]
mean = df["age"].mean()
std = df["age"].std()
is_null = df["age"].isnull().sum()
rng = np.random.default_rng(42)
rand_age = rng.integers(int(mean - std), int(mean + std), size=is_null)
df.loc[df["age"].isnull(), "age"] = rand_age

common_value = "S"
df["embarked"] = df["embarked"].fillna(common_value)
# fare thiếu 1 giá trị -> điền trung vị
df["fare"] = df["fare"].fillna(df["fare"].median())

# Chuyển giá vé về kiểu nguyên
df["fare"] = df["fare"].astype(int)

# Trích xuất danh hiệu (title) từ tên
titles = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
df["title"] = df["name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["title"] = df["title"].replace(["Lady", "Countess", "Capt", "Col", "Don", "Dr",
                                   "Major", "Rev", "Sir", "Jonkheer", "Dona"], "Rare")
df["title"] = df["title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df["title"] = df["title"].map(titles).fillna(0).astype(int)
df = df.drop(columns=["name"])

# Giới tính: 0 = nam, 1 = nữ
df["sex"] = df["sex"].map({"male": 0, "female": 1})
# Cổng lên tàu -> số
ports = {"S": 0, "C": 1, "Q": 2}
df["embarked"] = df["embarked"].map(ports)

# Tuổi -> số nguyên và chia thành 8 nhóm
df["age"] = df["age"].astype(int)
df["age_group"] = pd.cut(df["age"], bins=[-1, 11, 18, 22, 27, 33, 40, 66, np.inf],
                         labels=[0, 1, 2, 3, 4, 5, 6, 7]).astype(int)

# Giá vé -> 6 nhóm
df["fare_group"] = pd.cut(df["fare"], bins=[-1, 7.91, 14.454, 31, 99, 250, np.inf],
                          labels=[0, 1, 2, 3, 4, 5]).astype(int)

### Câu 8: Tạo thuộc tính mới age_class và fare_per_person

In [4]:
# Tuổi * loại vé
df["age_class"] = df["age"] * df["pclass"]
# Giá vé trên mỗi người (làm trơn +1)
df["fare_per_person"] = (df["fare"] / (df["relatives"] + 1)).astype(int)

In [5]:
df[["age", "pclass", "age_class", "fare", "relatives", "fare_per_person"]].head(10)

,age,pclass,age_class,fare,relatives,fare_per_person
0,29,1,29,211,0,211
1,0,1,0,151,3,37
2,2,1,2,151,3,37
3,30,1,30,151,3,37
4,25,1,25,151,3,37
5,48,1,48,26,0,26
6,63,1,63,77,1,38
7,39,1,39,0,0,0
8,53,1,53,51,2,17
9,71,1,71,49,0,49


In [6]:
# Bổ sung: chuẩn hóa Min-max cho age và fare về [0, 1]
df_norm = df.copy()
for col in ["age", "fare"]:
    df_norm[col + "_minmax"] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
df_norm[["age", "age_minmax", "fare", "fare_minmax"]].head(10)

,age,age_minmax,fare,fare_minmax
0,29,0.3625,211,0.412109
1,0,0.0000,151,0.294922
2,2,0.0250,151,0.294922
3,30,0.3750,151,0.294922
4,25,0.3125,151,0.294922
5,48,0.6000,26,0.050781
6,63,0.7875,77,0.150391
7,39,0.4875,0,0.000000
8,53,0.6625,51,0.099609
9,71,0.8875,49,0.095703


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   pclass           1309 non-null   int64
 1   survived         1309 non-null   int64
 2   sex              1309 non-null   int64
 3   age              1309 non-null   int64
 4   sibsp            1309 non-null   int64
 5   parch            1309 non-null   int64
 6   fare             1309 non-null   int64
 7   embarked         1309 non-null   int64
 8   relatives        1309 non-null   int64
 9   not_alone        1309 non-null   int64
 10  deck             1309 non-null   int64
 11  title            1309 non-null   int64
 12  age_group        1309 non-null   int64
 13  fare_group       1309 non-null   int64
 14  age_class        1309 non-null   int64
 15  fare_per_person  1309 non-null   int64
dtypes: int64(16)
memory usage: 163.8 KB


### Nhận xét
- age_class = age * pclass, fare_per_person = fare / (relatives + 1).
- Bổ sung chuẩn hóa Min-max đưa age và fare về đoạn [0, 1] để các thuộc tính có cùng thang đo.